# Lab 06: Parsing Static Web Pages

*INFO 4617/5617 · Week 6 · Chapter 6, [Parsing Static Web Pages](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-06-static-pages.html)*

This week every class is live coding: one lab in three parts, one a day. Each step says what we're building and why. Its code cells hold a comment saying what to write, and we write the code together. Cells that start with `# Given:` are already written: we read them together and run them. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over; click it to open it.

We turn four shapes of web page into DataFrames, the chapter's movies all the way, and each new tool gets a small try first, on a few lines of HTML you can read in full:
- **Part 1, Monday:** ask first, then find the table. Each site's `robots.txt`; The Numbers' box office chart for the last week of 2018, which ships a decoy table; and Wikipedia's [List of highest-grossing films](https://en.wikipedia.org/wiki/List_of_highest-grossing_films) in one `read_html()` call.
- **Part 2, Wednesday:** clean a table, and read an infobox. The list's gross column, cleaned and checked on a plot; then *The Godfather*'s infobox in English, French and Spanish.
- **Part 3, Friday:** cards to rows, and where the patterns stop. The Oscars' award cards across three ceremonies; Rotten Tomatoes' custom tags; and a page that JavaScript builds.

**Before class:** open this notebook in the `webdata` environment (week 1), and have your email address ready for the User-Agent. On Friday, open the [2026 ceremony page](https://www.oscars.org/oscars/ceremonies/2026) in Chrome next to this week's handout, *Cards to rows*: we Inspect it in Step 11.

On Wednesday and Friday, start at that day's setup step.

> **Instructor key.** PLACEHOLDER

# Part 1 — Monday: ask first, then find the table

Before we parse a page, we ask whether we may fetch it. Then two tables: The Numbers' weekly chart, which hides its data among two `<table>`s, and Wikipedia's list of highest-grossing films, which `read_html()` turns into a DataFrame in one call. Each starts with a few lines of HTML small enough to read in full.

## Step 0 — Set up (3 minutes)

Put your email address in `CLIENT`, and run the given cell. `HEADERS` sends the book's [User-Agent](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/User-Agent), `WebDataScience/1.0 (your-email@colorado.edu)`: week 2 explained why it names you. Wikimedia's [User-Agent policy](https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy) also asks for the library and its version, so `WIKI_HEADERS` adds `python-requests/` and its version, for Wikipedia. Why not one for every site? Step 3 shows. `DATA` is the course's folder of saved copies, the *checkpoints*.

In [ ]:
# Given: Part 1's setup. Put your email address in CLIENT, then run it
import time
from io import StringIO
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd
from bs4 import BeautifulSoup

CLIENT = "WebDataScience/1.0 (your-email@colorado.edu)"
HEADERS = {"User-Agent": CLIENT}
WIKI_HEADERS = {"User-Agent": CLIENT + " python-requests/" + requests.__version__}
DATA = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-06/data/"
print(WIKI_HEADERS["User-Agent"])

<details>
<summary>💬 Discuss</summary>

- In week 2 we put a contact address in our User-Agent. Who reads it, and what can they do with it?
- The chapter says oscars.org turns away a User-Agent that pretends to be Chrome but lets in one that names you. Week 5 showed how easy spoofing is: why would it get you less here, not more?

</details>

## Step 1 — Check before you scrape (10 minutes)

Before we send a page one request, ask its site's [`robots.txt`](https://www.rfc-editor.org/rfc/rfc9309.html), as week 2 did. First on two lines we write ourselves: a [`RobotFileParser`](https://docs.python.org/3/library/urllib.robotparser.html#urllib.robotparser.RobotFileParser) reads a list of lines with `parse()`, and `can_fetch(agent, url)` answers `True` or `False`. These two lines say every crawler (`*`) must stay out of paths that start `/w/`. Which of `https://en.wikipedia.org/w/index.php` and `https://en.wikipedia.org/wiki/The_Godfather` may we fetch? Then Wikipedia's real file, by hand, and a given loop for the chapter's six sites.

In [ ]:
# rp: a RobotFileParser(); parse the list ["User-agent: *", "Disallow: /w/"]
rp = RobotFileParser()
rp.parse(["User-agent: *", "Disallow: /w/"])

In [ ]:
# Print rp.can_fetch(CLIENT, url) for each of the two addresses above
print(rp.can_fetch(CLIENT, "https://en.wikipedia.org/w/index.php"))
print(rp.can_fetch(CLIENT, "https://en.wikipedia.org/wiki/The_Godfather"))

In [ ]:
# wiki_robots: request "https://en.wikipedia.org/robots.txt" with HEADERS
# Print its status and its first 200 characters
wiki_robots = requests.get("https://en.wikipedia.org/robots.txt", headers=HEADERS)
print(wiki_robots.status_code, wiki_robots.text[:200])

In [ ]:
# wiki_rp: a RobotFileParser(); parse wiki_robots.text.splitlines()
# Print wiki_rp.can_fetch(CLIENT, "https://en.wikipedia.org/w/index.php")
wiki_rp = RobotFileParser()
wiki_rp.parse(wiki_robots.text.splitlines())
print(wiki_rp.can_fetch(CLIENT, "https://en.wikipedia.org/w/index.php"))

In [ ]:
# Given: the chapter's pages, in the order of its strategies (1, 2, 3, 6, 7, 8)
CHAPTER_PAGES = [
    "https://www.boxofficemojo.com/weekend/2018W52/",
    "https://www.the-numbers.com/box-office-chart/weekly/2018/12/28",
    "https://en.wikipedia.org/wiki/List_of_highest-grossing_films",
    "https://www.oscars.org/oscars/ceremonies/2026",
    "https://www.rottentomatoes.com/browse/movies_in_theaters/",
    "https://www.imdb.com/title/tt0068646/",
]

In [ ]:
# Given: your two Wikipedia cells, for each page. urlparse(page).netloc is its host (week 5);
# parsers keeps each host's parsed file, so the microsprint can ask it again
parsers = {}
for page in CHAPTER_PAGES:
    host = urlparse(page).netloc
    robots = requests.get("https://" + host + "/robots.txt", headers=HEADERS)
    parsers[host] = RobotFileParser()
    parsers[host].parse(robots.text.splitlines())
    print(host, "may fetch:", parsers[host].can_fetch(CLIENT, page))
    time.sleep(1)

### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** set `sprint_agent` to another crawler's name, `"Googlebot"` (Google's search crawler) or `"GPTBot"` (OpenAI's), and run the given cell. It asks the six files we parsed about that crawler, with no new requests.
**Share:** your crawler, and which of the six pages it may fetch that we may not.

In [ ]:
# Microsprint: sprint_agent: "Googlebot", or "GPTBot"
sprint_agent = "Googlebot"

In [ ]:
# Given: ask each site's parsed robots.txt about sprint_agent instead of CLIENT
for page in CHAPTER_PAGES:
    host = urlparse(page).netloc
    print(host, sprint_agent, "may fetch:", parsers[host].can_fetch(sprint_agent, page))

<details>
<summary>💬 Discuss</summary>

- Open boxofficemojo.com/robots.txt in your browser. Its first lines are comments about scraping, pointing to IMDb's Conditions of Use. In week 2's terms, which binds us: the `Disallow` line, the conditions, or neither?
- Typed without `.splitlines()`, `parse()` reads the file one character at a time, finds no rules, and answers `True` for every page, with no error. Which of today's cells would have caught that?
- `robots.txt` speaks to crawlers. Is one request that a person sends from a notebook crawling? Where would you draw the line?

</details>

## Step 2 — One row, cell by cell (4 minutes)

Strategy 2's lesson, on a row you can read in full: three cells, the second empty, as a box-office row with no previous rank would have. A tag's `.text` is all the text inside it, run together. [`find_all("td")`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#find-all) gives each cell, the empty one too. The given cell then makes a list of each cell's text with a *list comprehension*: `[cell.get_text(strip=True) for cell in tiny_cells]` runs `get_text()` once for each cell and collects the results. What should each of the three show?

In [ ]:
# tiny_row: BeautifulSoup() of "<tr><td>1</td><td></td><td>Aquaman</td></tr>", with "html.parser"
# Print tiny_row.text
tiny_row = BeautifulSoup("<tr><td>1</td><td></td><td>Aquaman</td></tr>", "html.parser")
print(tiny_row.text)

In [ ]:
# tiny_cells: tiny_row.find_all("td"); print how many
tiny_cells = tiny_row.find_all("td")
print(len(tiny_cells))

In [ ]:
# Given: each cell's text, in a list. strip=True trims spaces and new lines from each end
[cell.get_text(strip=True) for cell in tiny_cells]

<details>
<summary>💬 Discuss</summary>

- `tiny_row.text` gave one string with no gap where the empty cell was. If you split it to get the cells, which column would `Aquaman` land in?
- The chapter calls `row.text.split("\n")` a tempting shortcut. Why does a missing cell shift every value after it, and would anything warn you?

</details>

## Step 3 — The table hiding among decoys (14 minutes)

Strategy 2. Step 1 found that Box Office Mojo and IMDb (one company owns both) disallow every crawler they don't name, so we leave them alone. The Numbers' chart for the week ending December 28, 2018 has the same kind of data, in two `<table>`s: one for wide screens, one for phones. Ask for it with `WIKI_HEADERS`, then `HEADERS`. Count the tables with `find_all("table")`, read one row by hand, and let the given cell do Step 2 for every row. See the chapter's ["Strategy 2"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-06-static-pages.html#strategy-2-the-table-hiding-among-decoys).

In [ ]:
# NUMBERS_URL: "https://www.the-numbers.com/box-office-chart/weekly/2018/12/28"
# refused: request NUMBERS_URL with WIKI_HEADERS; print its status and first 200 characters
NUMBERS_URL = "https://www.the-numbers.com/box-office-chart/weekly/2018/12/28"
refused = requests.get(NUMBERS_URL, headers=WIKI_HEADERS)
print(refused.status_code, refused.text[:200])

In [ ]:
# numbers_response: request NUMBERS_URL with HEADERS; print the status and the first 200 characters
numbers_response = requests.get(NUMBERS_URL, headers=HEADERS)
print(numbers_response.status_code, numbers_response.text[:200])

In [ ]:
# Given, only if the cell above didn't print 200: remove the "#" from the next line, and run this cell.
# It loads the page as the key's run saved it on 2026-11-17
# numbers_response = requests.get(DATA + "the-numbers-2018-12-28.html", headers=HEADERS)

In [ ]:
# numbers_soup: parse numbers_response.text with "html.parser"
# numbers_tables: numbers_soup.find_all("table"); print how many
numbers_soup = BeautifulSoup(numbers_response.text, "html.parser")
numbers_tables = numbers_soup.find_all("table")
print(len(numbers_tables))

In [ ]:
# Given: each table's position, its class, and its number of rows (<tr>). enumerate() numbers
# the tables as the loop goes, so "for i, table in" gets the number and the table each time
for i, table in enumerate(numbers_tables):
    print(i, table.get("class"), len(table.find_all("tr")), "rows")

In [ ]:
# desktop: the table with class_="chart-desktop"; desktop_rows: its find_all("tr")
desktop = numbers_soup.find("table", class_="chart-desktop")
desktop_rows = desktop.find_all("tr")

In [ ]:
# aquaman_cells: desktop_rows[1].find_all(["td", "th"])
# Print how many cells, and the third cell's get_text(strip=True)
aquaman_cells = desktop_rows[1].find_all(["td", "th"])
print(len(aquaman_cells), aquaman_cells[2].get_text(strip=True))

In [ ]:
# Given: the chapter's loop. header holds the first row's cell texts. Each later row is kept
# if it has as many cells as header; "if ... else" prints the others as skipped instead
header = [cell.get_text(strip=True) for cell in desktop_rows[0].find_all(["td", "th"])]
all_rows = []
for row in desktop_rows[1:]:
    values = [cell.get_text(strip=True) for cell in row.find_all(["td", "th"])]
    if len(values) == len(header):
        all_rows.append(values)
    else:
        print("Skipping a row with", len(values), "cells:", values[:3])
weekly = pd.DataFrame(all_rows, columns=header)
print(weekly.shape)
weekly.head()

### ⏱ Microsprint (5 minutes, alone or in pairs)

**Do:** the chapter takes `tables[1]`, the phone table, `chart-mobile`. Set `sprint_parser` to `"lxml"`, another of BeautifulSoup's [parsers](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#differences-between-parsers), and run the given cell. It parses the page with `"html.parser"` and with yours, and counts the phone table's rows, and the cells in each of its first three rows.
**Share:** how many cells the first data row has under each parser, and how many rows the chapter's loop would keep from it.

In [ ]:
# Microsprint: sprint_parser: "lxml"
sprint_parser = "lxml"

In [ ]:
# Given: the phone table under "html.parser" and under sprint_parser: its rows, and the
# number of cells in each of its first three rows
for parser in ["html.parser", sprint_parser]:
    mobile = BeautifulSoup(numbers_response.text, parser).find("table", class_="chart-mobile")
    mobile_rows = mobile.find_all("tr")
    first_three = [len(row.find_all(["td", "th"])) for row in mobile_rows[:3]]
    print(parser, len(mobile_rows), "rows; cells in the first three:", first_three)

<details>
<summary>💬 Discuss</summary>

- The Numbers refused `WIKI_HEADERS` with a 403 and accepted `HEADERS`; the only difference is `python-requests/` at the end. Week 5 said a 403 is often about the User-Agent: what is this server filtering on, and is that a reason to drop the library's name everywhere?
- Both tables have 84 rows, so the chapter's rule of thumb, "the main data table is usually the largest one", can't choose. What did, and where would week 5's Inspector have shown it to you?
- The phone table never closes its `<tr>` tags. Why do the two parsers disagree about it, and is either one wrong?

</details>

## Step 4 — One call for a whole table: `read_html()` (8 minutes)

Strategies 1 and 3. When the data sits in a real `<table>`, [`pd.read_html()`](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html) does Step 3's work in one call, and returns every table it finds as a *list* of DataFrames. Try it on one small table first. Wrap the text in [`StringIO`](https://docs.python.org/3/library/io.html#io.StringIO): pandas 3 reads a bare string as a file name. Then one revision of Wikipedia's list, `https://en.wikipedia.org/wiki/List_of_highest-grossing_films?oldid=1381243185`, fetched with `WIKI_HEADERS`. (Its [Permanent link](https://en.wikipedia.org/wiki/Help:Permanent_link) uses `/w/index.php`, which Step 1 found disallowed.)

In [ ]:
# Given: a table of one film, small enough to read in full. Two strings side by side in
# parentheses are joined into one
tiny_html = ("<table><tr><th>Film</th><th>Year</th></tr>"
             "<tr><td>Titanic</td><td>1997</td></tr></table>")

In [ ]:
# tiny_tables: pd.read_html(StringIO(tiny_html)); print how many tables, then show the first
tiny_tables = pd.read_html(StringIO(tiny_html))
print(len(tiny_tables))
tiny_tables[0]

In [ ]:
# WIKI_URL: the address above, with ?oldid=1381243185
# wiki_response: request WIKI_URL with WIKI_HEADERS; print its status and first 200 characters
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films?oldid=1381243185"
wiki_response = requests.get(WIKI_URL, headers=WIKI_HEADERS)
print(wiki_response.status_code, wiki_response.text[:200])

In [ ]:
# wiki_tables: pd.read_html on StringIO(wiki_response.text), with header=0; print how many tables
wiki_tables = pd.read_html(StringIO(wiki_response.text), header=0)
print(len(wiki_tables))

In [ ]:
# films: the first table; show its head()
films = wiki_tables[0]
films.head()

In [ ]:
# Given, only if the cells above stopped (Wikipedia can answer a busy classroom with 429):
# remove the "#" from the next line, and run this cell. It loads films as the key's run saved it on 2026-11-17
# films = pd.read_csv(DATA + "films.csv")

<details>
<summary>💬 Discuss</summary>

- `read_html` found 92 tables on one page, and we took the first on faith. Its `match=` argument (see its documentation) would choose for us: by what?
- On the small table you could predict every value before you ran it. What did that tell you about `read_html` that the 92 tables couldn't?
- Week 3 said an access matrix run once is a snapshot. What does the `oldid` record about our data that the plain URL doesn't?

</details>

## End of Part 1

On Wednesday, Part 2 cleans the list's gross column, finds the one value the chapter's cleanup gets wrong, and reads *The Godfather*'s infobox in three languages. It starts with its own setup step, and loads today's table from the course's repository.

# Part 2 — Wednesday: clean a table, and read an infobox

Today, two things a table can't do for you. Wikipedia's list of highest-grossing films parses in one call, but its gross column is text: we clean it, plot it, and find the one number the chapter's cleanup gets wrong. Then a film's infobox, one record laid out as label and value, in English, then French and Spanish, where the anchor that worked in English is gone.

## Step 5 — Set up again (3 minutes)

Part 2 starts fresh, so a student who missed Monday can start here. Put your email address in `CLIENT`, and run the two given cells. The second loads Monday's checkpoint, `films.csv`: the table `read_html()` gave us in Step 4.

In [ ]:
# Given: Part 2's setup. Put your email address in CLIENT, then run it
import time
from io import StringIO

import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

CLIENT = "WebDataScience/1.0 (your-email@colorado.edu)"
HEADERS = {"User-Agent": CLIENT}
WIKI_HEADERS = {"User-Agent": CLIENT + " python-requests/" + requests.__version__}
DATA = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-06/data/"

In [ ]:
# Given: Monday's table, Wikipedia's list of highest-grossing films (revision 1381243185),
# as the key's run saved it on 2026-11-17
films = pd.read_csv(DATA + "films.csv")
films.head()

<details>
<summary>💬 Discuss</summary>

- This table came from a file the key saved, not from Wikipedia. What can a saved copy not tell you that the live request can?

</details>

## Step 6 — Clean the gross column, and plot it to check (12 minutes)

[Strategy 3's cleanup](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-06-static-pages.html#cleaning-up-read_html-results). `Worldwide gross` is text, with a `$` and commas. The chapter removes every character that isn't a digit with [`str.replace()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.replace.html) and a *regular expression*, a pattern that describes text: `[^0-9]` means "one character that isn't 0 to 9". Try it on two values you can check by eye, then on the column, and convert the result with [`pd.to_numeric(errors="coerce")`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html). Then plot it: a number wrong by a factor of ten hides in a table and stands out on a plot.

In [ ]:
# Given: the chapter's pattern, named. r"..." keeps the backslashes and brackets as typed
NOT_A_DIGIT = r"[^0-9]"

In [ ]:
# two_grosses: pd.Series(["$2,923,710,708", "T$2,257,906,828"])
# Print two_grosses.str.replace(NOT_A_DIGIT, "", regex=True)
two_grosses = pd.Series(["$2,923,710,708", "T$2,257,906,828"])
print(two_grosses.str.replace(NOT_A_DIGIT, "", regex=True))

In [ ]:
# cleaned: the same replace on films["Worldwide gross"]
# films["gross"]: pd.to_numeric(cleaned, errors="coerce"); show films[["Title", "gross"]].head()
cleaned = films["Worldwide gross"].str.replace(NOT_A_DIGIT, "", regex=True)
films["gross"] = pd.to_numeric(cleaned, errors="coerce")
films[["Title", "gross"]].head()

In [ ]:
# Given: the chapter counts the footnote markers in Ref, "[", instead of throwing them away
# (in a pattern, "\[" means a real "["); then gross in billions against the year, on a plot
films["footnote_count"] = films["Ref"].str.count(r"\[")
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(films["Year"], films["gross"] / 1e9)
ax.set_title("The 50 highest-grossing films after the chapter's cleanup\n(Wikipedia, revision 1381243185)")
ax.set_xlabel("Release year")
ax.set_ylabel("Worldwide gross (billions of US dollars)")
plt.show()

### ⏱ Microsprint (5 minutes, alone or in pairs)

**Do:** find the point that doesn't belong. Sort `films` by `gross`, largest first, with [`sort_values()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html), and look at the raw `Worldwide gross` cell beside the cleaned number.
**Share:** the film, what its cell says, what the cleanup turned it into, and why.

In [ ]:
# Microsprint: by_gross: films.sort_values("gross", ascending=False)
# Show by_gross[["Title", "Worldwide gross", "gross"]].head(3)
by_gross = films.sort_values("gross", ascending=False)
by_gross[["Title", "Worldwide gross", "gross"]].head(3)

<details>
<summary>💬 Discuss</summary>

- The chapter's pattern keeps every digit. Why did that work for Titanic's `T$` and fail for this film?
- `errors="coerce"` turns what it can't read into NaN. Why couldn't it help here, and what kind of mistake can no setting catch?
- Would `films.head()` have shown you the problem? What does that say about how you check a column?

</details>

## Step 7 — Fix the cleanup, and count what changed (7 minutes)

The stray letters always come before the `$`, so take only what follows it. The given cell does that with [`str.extract()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html) and the pattern `\$([\d,]+)`: a `$`, then the digits and commas after it, which the [parentheses](https://docs.python.org/3/howto/regex.html#grouping) keep. Remove the commas (a plain `str.replace`, no pattern), convert, and compare with Step 6: how many rows changed?

In [ ]:
# Given: the digits and commas after the "$" in each cell. [0] is the part in parentheses
AFTER_DOLLAR = r"\$([\d,]+)"
after_dollar = films["Worldwide gross"].str.extract(AFTER_DOLLAR)[0]
after_dollar.head(3)

In [ ]:
# no_commas: after_dollar.str.replace(",", "")
# films["gross_fixed"]: pd.to_numeric(no_commas, errors="coerce")
no_commas = after_dollar.str.replace(",", "")
films["gross_fixed"] = pd.to_numeric(no_commas, errors="coerce")

In [ ]:
# changed: films["gross"] != films["gross_fixed"]
# Show films[changed], with the columns Title, "Worldwide gross", gross and gross_fixed
changed = films["gross"] != films["gross_fixed"]
films[changed][["Title", "Worldwide gross", "gross", "gross_fixed"]]

<details>
<summary>💬 Discuss</summary>

- The fix assumes every cell has a `$`. What would it do with one that doesn't, and how would you find out?
- The titles still carry `†` for films in theaters. If we joined this table to Step 3's by title, as Strategy 7 suggests, what would happen?
- The stray letters come from the article's own markup. Is fixing our code enough, or should someone fix the article too?

</details>

## Step 8 — An infobox: one record, label by label (11 minutes)

Strategy 4. A film's [infobox](https://en.wikipedia.org/wiki/Help:Infobox) is one record laid out as rows of a label (`<th>`) and a value (`<td>`). `read_html` would flatten it, so we take its rows apart with BeautifulSoup. In English, the table has the class `infobox`. Read one row by hand: which label and value should the third row hold? Then the given function, `read_infobox()`, does the same for every row, keeping a row only when it has both cells. See the chapter's ["Strategy 4"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-06-static-pages.html#strategy-4-the-infobox-pattern).

In [ ]:
# EN_URL: "https://en.wikipedia.org/wiki/The_Godfather"
# en_response: request EN_URL with WIKI_HEADERS; print its status and first 200 characters
EN_URL = "https://en.wikipedia.org/wiki/The_Godfather"
en_response = requests.get(EN_URL, headers=WIKI_HEADERS)
print(en_response.status_code, en_response.text[:200])

In [ ]:
# en_soup: parse en_response.text; infobox: its find("table", class_="infobox")
# infobox_rows: infobox.find_all("tr"); print how many
en_soup = BeautifulSoup(en_response.text, "html.parser")
infobox = en_soup.find("table", class_="infobox")
infobox_rows = infobox.find_all("tr")
print(len(infobox_rows))

In [ ]:
# director_row: infobox_rows[2]; print its find("th").get_text(strip=True),
# then its find("td").get_text(" ", strip=True)
director_row = infobox_rows[2]
print(director_row.find("th").get_text(strip=True))
print(director_row.find("td").get_text(" ", strip=True))

In [ ]:
# Given: the chapter's loop, as a function: your last cell, for every row of a table. A row with
# no <th> or no <td> gets None from find(), and "if label and value" skips it
def read_infobox(table):
    facts = {}
    for row in table.find_all("tr"):
        label = row.find("th")
        value = row.find("td")
        if label and value:
            facts[label.get_text(strip=True)] = value.get_text(" ", strip=True)
    return facts

facts = read_infobox(infobox)
print(len(facts), "facts. Directed by:", facts["Directed by"])
print(list(facts))

<details>
<summary>💬 Discuss</summary>

- One English key reads `Productioncompanies`, with no space. Where did the space go? (Look at that label in the Inspector.)
- `facts["Starring"]` runs the whole cast together in one string. What would you have to do before counting films per actor, and what could go wrong?

</details>

## Step 9 — The same function in French (13 minutes)

Strategy 5. In French, the class you'd see in the Inspector, `infobox_v3`, sits on a `<div>` around the table, so the chapter anchors on the table's own `<caption>` with a CSS selector: [`select("table:has(caption)")`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#css-selectors-through-the-css-property) finds every `<table>` that has a `<caption>` inside it, and `select("table.infobox_v3")` every `<table>` with that class. Try both on a few lines shaped like the French page first: how many matches should each give? Then the real page, and Step 8's function on it. See the chapter's ["Strategy 5"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-06-static-pages.html#strategy-5-porting-the-pattern-to-a-new-site).

In [ ]:
# Given: a few lines shaped like the French infobox: the class is on the <div>, not on the <table>
tiny_fr = BeautifulSoup(
    '<div class="infobox_v3"><table><caption>Le Parrain</caption>'
    '<tr><th>Réalisation</th><td>Coppola</td></tr></table></div>',
    "html.parser",
)

In [ ]:
# Print how many matches tiny_fr.select() finds for "table.infobox_v3", then for "table:has(caption)"
print(len(tiny_fr.select("table.infobox_v3")))
print(len(tiny_fr.select("table:has(caption)")))

In [ ]:
# FR_URL: "https://fr.wikipedia.org/wiki/Le_Parrain_(film)"
# fr_response: request FR_URL with WIKI_HEADERS; print its status and first 200 characters
FR_URL = "https://fr.wikipedia.org/wiki/Le_Parrain_(film)"
fr_response = requests.get(FR_URL, headers=WIKI_HEADERS)
print(fr_response.status_code, fr_response.text[:200])

In [ ]:
# fr_soup: parse fr_response.text; print how many matches it has for each of the two selectors
fr_soup = BeautifulSoup(fr_response.text, "html.parser")
print(len(fr_soup.select("table.infobox_v3")))
print(len(fr_soup.select("table:has(caption)")))

In [ ]:
# fr_infobox: the first match for "table:has(caption)"; facts_fr: read_infobox(fr_infobox)
# Print how many facts, and facts_fr["Réalisation"]
fr_infobox = fr_soup.select("table:has(caption)")[0]
facts_fr = read_infobox(fr_infobox)
print(len(facts_fr), facts_fr["Réalisation"])

### ⏱ Microsprint (5 minutes, alone or in pairs)

A short version of Additional Exercise 3, with an infobox instead of a table.

**Do:** port `read_infobox()` to a third edition. Set `sprint_url` to Spanish Wikipedia's [El padrino (película)](https://es.wikipedia.org/wiki/El_padrino_(pel%C3%ADcula)), and run the given cell. It counts each selector's matches, and reads the first `table.infobox` match.
**Share:** the selector that reached the infobox, how many facts it read, and the label Spanish uses for the director.

In [ ]:
# Microsprint: sprint_url: "https://es.wikipedia.org/wiki/El_padrino_(película)"
sprint_url = "https://es.wikipedia.org/wiki/El_padrino_(película)"

In [ ]:
# Given: fetch sprint_url, count both selectors' matches, and read the first table.infobox
sprint_response = requests.get(sprint_url, headers=WIKI_HEADERS)
sprint_soup = BeautifulSoup(sprint_response.text, "html.parser")
print(sprint_response.status_code)
print(len(sprint_soup.select("table.infobox")), "matches for table.infobox")
print(len(sprint_soup.select("table:has(caption)")), "matches for table:has(caption)")
facts_sprint = read_infobox(sprint_soup.select("table.infobox")[0])
print(len(facts_sprint), "facts:", list(facts_sprint))

<details>
<summary>💬 Discuss</summary>

- In week 5 the element picker labeled the CU Boulder article's infobox `table#mwCQ.infobox.vcard`: the class was on the table. On French Wikipedia, which element carries it, and why did that break `table.infobox_v3`?
- The French page has two tables with a caption, and we took the first. What is the second, and what would `read_infobox()` have returned from it?
- The three editions give different numbers of facts about the same film. Which one is the record of the film, if any?

</details>

## End of Part 2

On Friday, Part 3 turns the Oscars' award cards into rows, for three ceremonies; then reads Rotten Tomatoes' custom tags, and a page whose quotes JavaScript draws. It starts with its own setup step.

# Part 3 — Friday: cards to rows, and where the patterns stop

Today the data isn't in a table at all. The Oscars' ceremony page is a grid of *cards*, nested `<div>`s, one card per award: we take one card apart by hand, then a given loop turns every card into rows, for three ceremonies. Then two shapes where the patterns stop: Rotten Tomatoes' tags of its own, and a page that arrives without its data.

## Step 10 — Set up again (3 minutes)

Part 3 starts fresh. Put your email address in `CLIENT`, and run the given cell. Today needs no table from Monday or Wednesday.

In [ ]:
# Given: Part 3's setup. Put your email address in CLIENT, then run it
import time

import requests
import pandas as pd
from bs4 import BeautifulSoup

CLIENT = "WebDataScience/1.0 (your-email@colorado.edu)"
HEADERS = {"User-Agent": CLIENT}
DATA = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-06/data/"

<details>
<summary>💬 Discuss</summary>

- Part 3 sends only `HEADERS`, never `WIKI_HEADERS`. Which of this week's sites asked for the library's name, and which refused it?

</details>

## Step 11 — Cards that need to become rows: the Oscars (13 minutes)

Strategy 6. The page for the 98th Academy Awards has no `<table>`. Each award is a card of nested `<div>`s, with one block per honoree inside, winner first. oscars.org is built with Drupal, whose [class names](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#searching-by-css-class) describe Drupal's fields, not awards; the given cell names them. Find the area and its cards; take the first card apart; then read its first honoree's three fields, one a cell. The handout *Cards to rows* has the DevTools figures. See the chapter's ["Strategy 6"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-06-static-pages.html#strategy-6-cards-that-need-to-become-rows).

In [ ]:
# OSCARS_URL: "https://www.oscars.org/oscars/ceremonies/2026"
# oscars_response: request OSCARS_URL with HEADERS; print its status and first 200 characters
OSCARS_URL = "https://www.oscars.org/oscars/ceremonies/2026"
oscars_response = requests.get(OSCARS_URL, headers=HEADERS)
print(oscars_response.status_code, oscars_response.text[:200])

In [ ]:
# Given: the Drupal class names, as the Inspector shows them, from the outside in
AREA = "field--name-field-award-categories"           # the div that holds every card
CATEGORY = "paragraph--type--award-category"          # one card: one award
CATEGORY_NAME = "field--name-field-award-category-oscars"
HONOREE = "paragraph--type--award-honoree"            # one nominee or winner in a card
HONOREE_TYPE = "field--name-field-honoree-type"       # "Winner" or "Nominees"
ENTITY = "field--name-field-award-entities"           # who
FILM = "field--name-field-award-film"                 # for which film

In [ ]:
# oscars_soup: parse oscars_response.text; area: its find("div", class_=AREA)
# categories: area.find_all("div", class_=CATEGORY); print how many
oscars_soup = BeautifulSoup(oscars_response.text, "html.parser")
area = oscars_soup.find("div", class_=AREA)
categories = area.find_all("div", class_=CATEGORY)
print(len(categories))

In [ ]:
# card: the first category; card_name: its find(class_=CATEGORY_NAME); print card_name's text
card = categories[0]
card_name = card.find(class_=CATEGORY_NAME)
print(card_name.get_text(strip=True))

In [ ]:
# honorees: card's divs with class_=HONOREE; print how many
honorees = card.find_all("div", class_=HONOREE)
print(len(honorees))

In [ ]:
# first: the first honoree; print the text of its find(class_=HONOREE_TYPE)
first = honorees[0]
print(first.find(class_=HONOREE_TYPE).get_text(strip=True))

In [ ]:
# Print the text of first's find(class_=ENTITY)
print(first.find(class_=ENTITY).get_text(strip=True))

In [ ]:
# film_tag: first's find(class_=FILM); print its text
film_tag = first.find(class_=FILM)
print(film_tag.get_text(strip=True))

In [ ]:
# Given: your last five cells, for every card and every honoree in it: one dict per honoree.
# Some honorees have no film (an honorary award goes to a person), so find() gives None:
# "honoree_film.get_text(...) if honoree_film else None" records None instead of stopping
results = []
for category in categories:
    category_name = category.find(class_=CATEGORY_NAME).get_text(strip=True)
    for honoree in category.find_all("div", class_=HONOREE):
        honoree_film = honoree.find(class_=FILM)
        results.append({
            "category": category_name,
            "honoree_type": honoree.find(class_=HONOREE_TYPE).get_text(strip=True),
            "entity": honoree.find(class_=ENTITY).get_text(strip=True),
            "film": honoree_film.get_text(strip=True) if honoree_film else None,
        })
oscars26 = pd.DataFrame(results)
oscars26["honoree_type"].value_counts()

### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** which film had the most nominations? Count the rows per film with [`value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html), and show the top five.
**Share:** the film, its count, and one category where a film's nomination doesn't show up under `film`.

In [ ]:
# Microsprint: film_counts: oscars26["film"].value_counts(); show its head()
film_counts = oscars26["film"].value_counts()
film_counts.head()

<details>
<summary>💬 Discuss</summary>

- 25 winners for 24 categories: a bug in our loop, or a fact about the night? How did the handout settle it?
- The column we named `film` holds a country for International Feature Film and a song's title for Original Song. Who decides what a column means, we or the site?
- Week 4's House XML named its contents in its tags, such as `<party>`. How good a guide are Drupal's class names, by comparison?

</details>

## Step 12 — One function, several ceremonies (8 minutes)

We want more years, so the given cell gathers Step 11's fetch, area and loop into the chapter's function, `parse_oscar_ceremony(year)`, with the year in the URL and a `"year"` column. Find your lines in it. Run it on one year by hand; then the given loop runs three, and [sleeps a second](https://docs.python.org/3/library/time.html#time.sleep) between them. The chapter's version fetches through week 2's `responsible_get()`, which returns `None` when a request fails; ours assumes every request works. See the chapter's ["Abstracting into Functions"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-06-static-pages.html#abstracting-into-functions).

In [ ]:
# Given: Step 11 as a function of the year. It returns one row per honoree
def parse_oscar_ceremony(year):
    response = requests.get("https://www.oscars.org/oscars/ceremonies/" + str(year), headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")
    results = []
    for category in soup.find("div", class_=AREA).find_all("div", class_=CATEGORY):
        category_name = category.find(class_=CATEGORY_NAME).get_text(strip=True)
        for honoree in category.find_all("div", class_=HONOREE):
            honoree_film = honoree.find(class_=FILM)
            results.append({
                "category": category_name,
                "honoree_type": honoree.find(class_=HONOREE_TYPE).get_text(strip=True),
                "entity": honoree.find(class_=ENTITY).get_text(strip=True),
                "film": honoree_film.get_text(strip=True) if honoree_film else None,
                "year": year,
            })
    return pd.DataFrame(results)

In [ ]:
# oscars_2024: parse_oscar_ceremony(2024); show its "honoree_type" value_counts()
oscars_2024 = parse_oscar_ceremony(2024)
oscars_2024["honoree_type"].value_counts()

In [ ]:
# Given: three ceremonies, a second apart, in one table; then each year's categories and honorees
frames = []
for year in [2024, 2025, 2026]:
    frames.append(parse_oscar_ceremony(year))
    time.sleep(1)
oscars = pd.concat(frames, ignore_index=True)
oscars.groupby("year").agg(categories=("category", "nunique"), honorees=("category", "size"))

In [ ]:
# Given, only if the cell above stopped: remove the "#" from the next line, and run this cell.
# It loads the three ceremonies as the key's run saved them on 2026-11-17
# oscars = pd.read_csv(DATA + "oscars-2024-2026.csv")

### ⏱ Microsprint (4 minutes, alone or in pairs)

A short version of Additional Exercise 2.

**Do:** set `sprint_year` to a ceremony from before you were born (the key uses 1990), and run the given cell. It counts that ceremony's categories and honorees, and shows the rows with no film.
**Share:** the year, its two counts, and the category of any row whose `film` is `None`.

In [ ]:
# Microsprint: sprint_year: a ceremony before you were born, such as 1990
sprint_year = 1990

In [ ]:
# Given: one more ceremony, and its rows with no film
old = parse_oscar_ceremony(sprint_year)
print(old["category"].nunique(), "categories,", len(old), "honorees")
old[old["film"].isna()][["category", "honoree_type"]]

<details>
<summary>💬 Discuss</summary>

- In 2024–2026 the film guard never fired. Which row made it fire in your year, and what would the loop have done without it?
- Our function assumes every request works and every page has the area div. What would it do with a 429 for 2025, or after a redesign, and how do the chapter's `responsible_get()` and `safe_scrape()` guard against each?
- If you parsed every ceremony since 1929, what would you check before comparing the number of categories or nominees across years?

</details>

## Step 13 — Where the patterns stop: custom tags, and a page built by JavaScript (8 minutes)

[Strategy 7](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-06-static-pages.html#strategy-7-inconsistent-cards-multiple-sites): Rotten Tomatoes' [Movies in Theaters](https://www.rottentomatoes.com/browse/movies_in_theaters/) grid is cards in the site's own tags, `<media-info-tile>` and `<rt-text>`, told apart by their `slot` attribute. BeautifulSoup reads them like any tag: try one small tile first. [Strategy 8](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-06-static-pages.html#strategy-8-when-static-isnt): some pages arrive nearly empty, and JavaScript fills them in the browser; `requests` never runs it. Step 1 ruled out the chapter's IMDb, so we use [quotes.toscrape.com](https://quotes.toscrape.com/), built for practice: the same quotes at `/` as HTML, and at `/js/` drawn by a script.

In [ ]:
# Given: one tile, as Rotten Tomatoes writes them
tiny_tile = BeautifulSoup(
    '<media-info-tile><rt-text slot="criticsScore">93%</rt-text></media-info-tile>',
    "html.parser",
)

In [ ]:
# tiny_score: tiny_tile.find("rt-text", attrs={"slot": "criticsScore"}); print its text
tiny_score = tiny_tile.find("rt-text", attrs={"slot": "criticsScore"})
print(tiny_score.get_text(strip=True))

In [ ]:
# RT_URL: "https://www.rottentomatoes.com/browse/movies_in_theaters/"
# rt_response: request RT_URL with HEADERS; print its status and first 200 characters
RT_URL = "https://www.rottentomatoes.com/browse/movies_in_theaters/"
rt_response = requests.get(RT_URL, headers=HEADERS)
print(rt_response.status_code, rt_response.text[:200])

In [ ]:
# Given, to see the key's grid, or if the cell above didn't print 200: remove the "#" from the next
# line, and run this cell. It loads the page as the key's run saved it on 2026-11-17 (the grid changes weekly)
# rt_response = requests.get(DATA + "rotten-tomatoes-in-theaters-2026-11-17.html", headers=HEADERS)

In [ ]:
# rt_soup: parse rt_response.text; tiles: its find_all("media-info-tile"); print how many
rt_soup = BeautifulSoup(rt_response.text, "html.parser")
tiles = rt_soup.find_all("media-info-tile")
print(len(tiles))

In [ ]:
# Given: every tile's title and Tomatometer, in a table. A film too new to have enough
# reviews has an empty criticsScore, so its score is ""
tile_rows = []
for tile in tiles:
    title = tile.find("rt-text", attrs={"data-qa": "discovery-media-list-item-title"})
    score = tile.find("rt-text", attrs={"slot": "criticsScore"})
    tile_rows.append({
        "title": title.get_text(strip=True),
        "tomatometer": score.get_text(strip=True),
    })
in_theaters = pd.DataFrame(tile_rows)
print((in_theaters["tomatometer"] == "").sum(), "of", len(in_theaters), "with no Tomatometer yet")
in_theaters.head()

In [ ]:
# quotes_response: request "https://quotes.toscrape.com/" with HEADERS
# Print its status and its first 200 characters
quotes_response = requests.get("https://quotes.toscrape.com/", headers=HEADERS)
print(quotes_response.status_code, quotes_response.text[:200])

In [ ]:
# quotes_soup: parse quotes_response.text; print how many matches it has for "div.quote"
quotes_soup = BeautifulSoup(quotes_response.text, "html.parser")
print(len(quotes_soup.select("div.quote")))

In [ ]:
# The same for "https://quotes.toscrape.com/js/": js_response, js_soup, and the number of "div.quote"
js_response = requests.get("https://quotes.toscrape.com/js/", headers=HEADERS)
js_soup = BeautifulSoup(js_response.text, "html.parser")
print(len(js_soup.select("div.quote")))

In [ ]:
# Given: the two pages side by side: characters, quotes, and <script> tags
for name, response, soup in [("/", quotes_response, quotes_soup), ("/js/", js_response, js_soup)]:
    print(name, len(response.text), "characters,", len(soup.select("div.quote")), "quotes,",
          len(soup.find_all("script")), "scripts")

<details>
<summary>💬 Discuss</summary>

- The `/js/` page came back with status 200 and a body, and no quotes in it. Week 5 said the Inspector shows the DOM, not what the server sent. Where are the quotes in what `requests` got? (Try View Source.)
- Place this week's sites on the chapter's 2×2 of single or multiple sites and consistent or inconsistent design. Which quadrant needs the most code?
- To ask whether a film's box office tracks its Tomatometer, you'd join The Numbers to Rotten Tomatoes by title. What have we seen this week that would break that join?

</details>

## Step 14 — What did we learn? (3 minutes)

In a few sentences, as comments or in a Markdown cell:

- Which of the chapter's sites did `robots.txt` let us fetch, and what did the two small lines of Step 1 predict about Wikipedia's real file?
- How many films did The Numbers' desktop table give us, and what would the chapter's `tables[1]` have given under each parser?
- What did the chapter's cleanup do to one film's gross, and what caught it?
- How many honorees did we collect across 2024–2026, and what did an older ceremony show about the film guard?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday). Choose a table other than the lab's list of highest-grossing films.

- Step 1, choose and check: Step 1's `RobotFileParser` on Wikipedia's file (week 2), and Step 4's fetch with `WIKI_HEADERS`.
- Step 2, cache the raw HTML: this lab's checkpoints are saved pages, read like live ones; Step 4 fetched once and parsed the text it kept (Additional Exercise 4 is the longer version).
- Step 3, extract with `read_html()`: Step 4's `read_html(StringIO(...))`, the table count, and the index.
- Step 4, clean it: Steps 6–7's cleanup, `pd.to_numeric(errors="coerce")`, and the check that found what it missed.
- Step 5, visualize: Step 6's titled, labeled plot (yours is a bar chart).
- Step 6, look under the hood: Step 3's `find_all("table")`, each table's `class`, and Steps 2–3's rows read with `find_all(["td", "th"])`.
- Step 7, interpret: Steps 3 and 9, where a table index and a class name broke.

In [ ]:
# Your answers here, as comments
# ANSWERS

## Friday: from the lab to the textbook (15 minutes)

What did the lab turn up that the chapter should know? Some places to start:

- **Step 1:** the chapter's Strategies 1 and 8 request pages whose `robots.txt` disallows every unnamed crawler. What should the chapter use, or say, instead?
- **Step 3:** the chapter takes `tables[1]` and prints a header with an empty column and a `Movie` column. Compare both with what you got.
- **Steps 6–7:** the chapter's pattern and The Fate of the Furious. Is a fix, a warning, or a check the better revision?
- **Steps 8–9:** the `Productioncompanies` key, or a third language edition as an example of porting.
- **A small first call** that would help the chapter's reader: a few lines of HTML before a real page.

[`handouts/common/revision-framework.md`](../../handouts/common/revision-framework.md) says how to propose a revision.